In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')
print(f'TensorLy tenalg backend: {tl.tenalg.get_backend()}')

env: CUPY_ACCELERATORS=cutensor,cub
TensorLy backend: cupy
TensorLy tenalg backend: einsum


In [2]:
from moabb.paradigms import FilterBankMotorImagery
from moabb.datasets import *
from mne.decoding import Scaler

from notebooks.data_loader import fh_envelope, postprocess



paradigm = FilterBankMotorImagery(resample=250)
dataset = AlexMI()
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
)
X = fh_envelope(X, sfreq=250, target_sfreq=32)
X = postprocess(X,y)

X.shape

Choosing from all possible events
/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MB, data loaded,
 'right_hand': 20
 'feet': 20>

/data/leuven/352/vsc35289/miniconda3/envs/hoda-bci/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 40 events (all good), 0 – 3 s (baseline off), ~7.5 MB, dat

(40, 6, 16, 96)

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import pandas as pd

x = X.flatten()
x = tl.to_numpy(x)
hist, bin_edges = np.histogram(x, bins=100000)
bins = (bin_edges[:-1] + bin_edges[1:]) / 2  # Compute the mean of each subsequent pair
df = pd.DataFrame({'bins':bins, 'count':hist})
px.histogram(df, x="bins", y="count")

In [4]:
from hoda.hoda import HODA

hoda = HODA(
        rank=None,
        max_iter=1024,
        tol=1e-6,
        shrinkage='lw',
        toeplitz=(2,),
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        forward=True,
        theta=0.2,
        refit_shrinkage=False,
)


In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('default')

hoda.fit_backward(X,y)
df = pd.DataFrame(hoda.train_info_['backward'])
display(df)

Backward HODA model rank=(1, 1, 3):  14%|█▍        | 144/1024 [00:02<00:17, 49.92it/s]


,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,8.389971e-01,0.286640,64139.449917
1,1,2,2,7.525335e-01,0.043704,85409.312030
2,1,3,3,1.520170e+00,0.319910,23047.434866
3,2,1,4,2.888390e-01,0.286640,63394.601929
4,2,2,5,4.678131e-01,0.043704,51547.457865
...,...,...,...,...,...,...
430,144,2,431,1.011940e-06,0.043704,14207.197675
431,144,3,432,8.705480e-07,0.319910,11542.485880
432,145,1,433,6.236438e-07,0.286640,61260.422604
433,145,2,434,8.161216e-07,0.043704,14207.191628


In [6]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

,iteration,mode,flip,update,shrinkage,objective
0,1,1,1,8.389971e-01,0.286640,64139.449917
1,1,2,2,7.525335e-01,0.043704,85409.312030
2,1,3,3,1.520170e+00,0.319910,23047.434866
3,2,1,3,2.888390e-01,0.286640,63394.601929
4,2,2,4,4.678131e-01,0.043704,51547.457865
...,...,...,...,...,...,...
430,144,2,288,1.011940e-06,0.043704,14207.197675
431,144,3,289,8.705480e-07,0.319910,11542.485880
432,145,1,289,6.236438e-07,0.286640,61260.422604
433,145,2,290,8.161216e-07,0.043704,14207.191628


In [7]:
import plotly.express as px

if hoda.extra_train_info:
    fig = px.line(df, x='flip', y='F_tr', log_y=False)
    fig.show()

In [8]:
px.line(df, x='iteration', y='update', log_y=True, color='mode')


In [9]:
px.line(df, x='flip', y='shrinkage', log_y=False, color='mode')

In [10]:
px.line(df, x='flip', y='objective', log_y=False, color='mode')

In [11]:
hoda.fit_forward(X,y)
df = pd.DataFrame(hoda.train_info_['forward'])
df

Forward model :   4%|▍         | 39/1023 [00:00<00:04, 208.83it/s]


,iteration,mode,flip,update,lambda_
0,1,1,1,3.219464e-01,0.0
1,1,2,2,1.339274e+00,0.0
2,1,3,3,1.365414e+01,0.0
3,2,1,4,3.256408e-01,0.0
4,2,2,5,6.131708e-01,0.0
...,...,...,...,...,...
115,39,2,116,1.740980e-07,0.0
116,39,3,117,1.093664e-06,0.0
117,40,1,118,7.368747e-07,0.0
118,40,2,119,1.265017e-07,0.0


In [12]:
df['flip'] = (df['iteration']-1)*2+df['mode']
df

,iteration,mode,flip,update,lambda_
0,1,1,1,3.219464e-01,0.0
1,1,2,2,1.339274e+00,0.0
2,1,3,3,1.365414e+01,0.0
3,2,1,3,3.256408e-01,0.0
4,2,2,4,6.131708e-01,0.0
...,...,...,...,...,...
115,39,2,78,1.740980e-07,0.0
116,39,3,79,1.093664e-06,0.0
117,40,1,79,7.368747e-07,0.0
118,40,2,80,1.265017e-07,0.0


In [13]:
if hoda.extra_train_info:
    x.line(df, x='flip', y='mse', log_y=True)

In [14]:
px.line(df, x='flip', y='update', log_y=True, color='mode')

In [15]:
from sklearn.feature_selection import SelectFpr, SelectFwe, SelectFdr
import numpy as np

Xt = hoda.transform(X)
xt = tl.to_numpy(tl.unfold(Xt,0))

select = SelectFdr(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})
df

,feature,F,p_value,significant
0,0,0.008449,9.272452e-01,False
1,1,0.069327,7.937413e-01,False
2,2,261.114907,1.294473e-18,True


In [16]:
fig = px.bar(df, x='feature', y='F', color='significant', log_y=True)
fig.update_xaxes(type='category', categoryorder='total descending')
fig.show()

In [17]:
from sklearn.decomposition import PCA
x_viz = PCA(n_components=2).fit_transform(xts)
px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)

ValueError: n_components=2 must be between 0 and min(n_samples, n_features)=1 with svd_solver='covariance_eigh'

In [ ]:
%load_ext autoreload
%autoreload 2

import plotly.io as pio
pio.renderers.default = 'iframe'
